# `NescienceClassifier` demo

This notebook demonstrates minimum-nescience classification with `NescienceClassifier`.

The classifier implements **minimum-nescience model selection**:

```python
fit candidate classifier on (X, y)
extract explicit artifacts:
    subset, predictions, model_string
compute nescience
select the candidate with minimum nescience
```

The class does **not** use a train/test split or cross-validation. The theory evaluates the model with respect to the available effective representation `(X, y)`. Excessive complexity is penalized internally through the nescience components, especially `surfeit` and `surplus`.

This demo covers:

- default candidate search;
- custom candidate lists;
- result tables;
- component plots;
- explanations;
- class predictions;
- probability predictions;
- canonical model-string inspection;
- weight sensitivity;
- DataFrame feature-name preservation;
- multiclass classification;
- string-label preservation;
- strict unsupported-model behavior.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier

from mnplib.classifier import NescienceClassifier
from pprint import pformat

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Helper functions

These helpers are only for notebook display. They are not part of the library.


In [ ]:
def plot_nescience(df, title):
    """Plot scalar nescience for candidate classifiers."""
    df.set_index("candidate")["nescience"].plot(kind="bar", figsize=(11, 4))
    plt.ylabel("Nescience")
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_components(df, title):
    """Plot the four nescience components for candidate classifiers."""
    component_cols = ["deficiency", "surplus", "inaccuracy", "surfeit"]
    df.set_index("candidate")[component_cols].plot(kind="bar", figsize=(12, 5))
    plt.ylabel("Component value")
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_confusion(y_true, y_pred, labels, title):
    """Plot a confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.show()


def print_section(title):
    """Print a visible separator."""
    print("\n" + title)
    print("=" * len(title))

## 3. Build a synthetic binary classification problem

The dataset has ten input features, but only four are informative. This is useful for seeing how different candidate classifiers trade off accuracy, feature use, and description length.


In [ ]:
X, y = make_classification(
    n_samples=800,
    n_features=10,
    n_informative=4,
    n_redundant=0,
    n_repeated=0,
    n_classes=2,
    class_sep=1.4,
    random_state=42,
)

feature_names = [f"x{i}" for i in range(X.shape[1])]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", np.unique(y))

## 4. Fit the default `NescienceClassifier`

The default candidate set currently includes:

- `LogisticRegression` with several values of `C`;
- `DecisionTreeClassifier` with several depth values.

All candidates are evaluated on the same representation `(X, y)`.


In [ ]:
clf = NescienceClassifier(
    random_state=42,
    verbose=1,
)

clf.fit(X, y)

## 5. Selected classifier and main diagnostics

In [ ]:
print("Best candidate:", clf.best_candidate_name_)
print("Selected estimator:", type(clf.model_).__name__)
print("Best nescience:", clf.nescience())
print("Native classifier score on full data:", clf.score(X, y))
print("Accuracy:", accuracy_score(y, clf.predict(X)))
print("Classes:", clf.classes_)

print("\nComponents:")
for name, value in clf.components().items():
    print(f"{name:>10}: {value:.6f}")

## 6. Candidate comparison table

`results_dataframe()` gives a compact table with the selected classifier first because rows are sorted by ascending nescience.


In [ ]:
results = clf.results_dataframe()
results

In [ ]:
plot_nescience(results, "Default candidates: scalar nescience")
plot_components(results, "Default candidates: nescience components")

## 7. Native predictive score versus nescience

The native estimator score for classifiers is usually accuracy, while nescience is a different objective. A classifier may have high accuracy but still have higher nescience because its description is unnecessarily complex or because it uses irrelevant information.


In [ ]:
comparison = results[
    [
        "candidate",
        "model_type",
        "native_estimator_score",
        "nescience",
        "inaccuracy",
        "surfeit",
        "n_selected_features",
        "description_length",
    ]
].copy()

comparison.sort_values("native_estimator_score", ascending=False)

In [ ]:
comparison.set_index("candidate")[["native_estimator_score", "nescience"]].plot(
    kind="bar",
    figsize=(11, 4),
)
plt.title("Native classifier score versus nescience")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Predictions, probabilities, and confusion matrix

In [ ]:
predictions = clf.predict(X)
probabilities = clf.predict_proba(X)

print("Predictions shape:", predictions.shape)
print("Probabilities shape:", probabilities.shape)
print("First 5 predictions:", predictions[:5])
print("First 5 probability rows:")
print(probabilities[:5])

plot_confusion(y, predictions, labels=clf.classes_, title="Selected classifier confusion matrix")

## 9. Explanation of the selected classifier

`analysis()` returns the selected candidate's metrics, effective features, and reported hyperparameters. `pformat()` presents the dictionary as a formatted numerical dictionary. The candidate-evaluation accuracy was recorded on training data, not a held-out set.


In [ ]:
explanation = clf.analysis()

print(pformat(explanation))

## 10. Inspect the canonical model string

The selected classifier is serialized into the canonical model-description schema used by the new scikit-learn adapter layer.


In [ ]:
model_string = clf.model_description()["model_string"]

print(model_string[:2500])
print("\nDescription length in bytes:", len(model_string.encode("utf-8")))

## 13. DataFrame Feature Names

Feature names are preserved in diagnostics. Canonical descriptions use stable feature tokens.


In [ ]:
X_df = pd.DataFrame(X, columns=[f"feature_{j}" for j in range(X.shape[1])])
df_clf = NescienceClassifier(
    models=["logistic_regression", "decision_tree"],
    random_state=42,
    search_options={"logistic_regression": {"max_iter": 1000}},
).fit(X_df, y)
print("Feature names:", list(df_clf.feature_names_in_))
print(df_clf.model_description()["model_string"][:1500])


## 11. Multiclass classification

The class also supports multiclass problems as long as the underlying model serializer supports the selected estimator.


In [ ]:
Xm, ym = make_classification(
    n_samples=900,
    n_features=12,
    n_informative=5,
    n_redundant=0,
    n_repeated=0,
    n_classes=3,
    n_clusters_per_class=1,
    class_sep=1.3,
    random_state=123,
)

multiclass_clf = NescienceClassifier(
    random_state=42
)

multiclass_clf.fit(Xm, ym)

print("Best candidate:", multiclass_clf.best_candidate_name_)
print("Classes:", multiclass_clf.classes_)
print("Accuracy:", multiclass_clf.score(Xm, ym))
print("Nescience:", multiclass_clf.nescience())

multiclass_clf.results_dataframe()

In [ ]:
plot_components(
    multiclass_clf.results_dataframe(),
    "Multiclass candidates: nescience components",
)

plot_confusion(
    ym,
    multiclass_clf.predict(Xm),
    labels=multiclass_clf.classes_,
    title="Multiclass selected classifier confusion matrix",
)

## 15. String-label preservation

The classifier preserves original labels. This is useful when target classes are semantic labels rather than integers.


In [ ]:
string_labels = np.array([f"class_{label}" for label in y])

string_clf = NescienceClassifier(
    random_state=42,
)

string_clf.fit(X, string_labels)

print("Stored classes:", string_clf.classes_)
print("First 10 predictions:")
print(string_clf.predict(X[:10]))

print("\nCanonical string excerpt:")
print(string_clf.model_description()["model_string"][:1500])

## 16. Strict unsupported-model behavior

Unsupported models should fail explicitly. This avoids silently using an incorrect feature subset or an incomparable model string.


In [ ]:
try:
    unsupported = NescienceClassifier(
    )
    unsupported.fit(X, y)
except NotImplementedError as exc:
    print(type(exc).__name__)
    print(exc)

## 17. Summary

The new `NescienceClassifier` is a small model-selection orchestrator:

```python
NescienceClassifier
    -> fits candidate classifiers on (X, y)
    -> extracts canonical artifacts with mnplib.models
    -> computes nescience with Nescience
    -> selects the candidate with minimum nescience
```

Its most useful outputs are:

```python
clf.best_candidate_name_
clf.best_nescience_
clf.components()
clf.analysis()
print(pformat(clf.analysis()))
clf.results_dataframe()
clf.model_description()["model_string"]
clf.predict(X)
clf.predict_proba(X)
```

This keeps the model-selection logic separate from the metric classes and from the model serializers.
